# **Taller: Clasificación de secuencias de nucleótidos usando modelos de lenguaje genómicos basados en _transformers_**
### Grupo de investigación del Dr. Carlos Francisco Méndez Cruz

### Taller ELIXIR-México de Recursos de Datos y Conocimiento

------------------------------------------------------------------------


# Objetivo del taller

Afinar un Modelo de Lenguaje Genómico de DNA (_transformer pre-entrenado_) llamado DNABERT-2 (Zhou et al., 2024) para predecir si una secuencia de nucleótidos es o no es un promotor de _E. coli_.

* Zhou, Zhihan, et al. "DNABERT-2: Efficient foundation model and
benchmark for multi-species genomes." _International Conference on Learning Representations_. Vol. 2024. 2024.

# Descripción de la tarea

El modelo debe clasificar secuencias de 57 nucleótidos de longitud en dos categorías:

0. no promotor
1. promotor

# Conjunto de datos
Usaremos el conjunto de datos llamado [Promoter gene sequences](https://archive.ics.uci.edu/dataset/67/molecular+biology+promoter+gene+sequences) con 106 secuencias de 57 nucleótidos de longitud formadas con los símbolos {a, g, t, c}.


------------------------------------------------------------------------


## Importación de módulos y librerías

In [ ]:
# Uso de transformers
from transformers import AutoTokenizer, BertForSequenceClassification

# Uso de redes neuronales
import torch
from torch import nn
import torch.nn.functional as F

# Procesamiento de datos
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# Evaluación del modelo
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Gráficos
import matplotlib.pyplot as plt


# 1. Manejo de datos


## 1.1. Subir archivo CSV a Colab
Usaremos el archivo promoters.csv que le hemos entregado.

## 1.2. Leer archivo CSV en Dataframe
* df_data

In [ ]:
df_data = pd.read_csv('promoters.csv')
print("Dimensionalidad de datos (renglones y columnas):", df_data.shape)
print("Categorías de secuencias:", df_data.Category.unique())
print("Promotores por categoría")
print(df_data.groupby(['Category']).size() )
df_data.head()


## 1.3. Preparación de datos
Pasamos las secuencias a mayúsculas porque así fue pre-entrenado DNABERT-2.

In [ ]:
df_data['Sequence'] = df_data['Sequence'].str.upper()
df_data.head()

## 1.4. Separación de datos en entrenamiento y prueba
1.   df_train
2.   df_test




In [ ]:
df_train, df_test = train_test_split(df_data, test_size=0.2, random_state=42, stratify=df_data.Category)
print("Dimensionalidad de datos de entrenamiento:", df_train.shape)
print("Instancias por categoría")
print(df_train.groupby(['Category']).size())

print("Dimensionalidad de datos de prueba:", df_test.shape)
print("Instancias por categoría")
print(df_test.groupby(['Category']).size())


# 2. Tokenizador de DNABERT-2
## 2.1. Descarga del tokenizador

In [ ]:
# Número máximo de tokens (fragmentos de secuencia)
max_tokens = 15

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)


## 2.2. Exploración del tokenizador

In [ ]:
# ¿Cuántos tokens únicos tiene el Modelo de Lenguaje Genómico DNABERT-2?
print("Tamaño del vocabulario del tokenizador (número de tokens únicos): {}".format(tokenizer.vocab_size))
# Obtenemos el vocabulario del tokenizador
vocabulario = tokenizer.get_vocab()
# Imprimimos primeras diez palabras del vocabulario
print("Primeros 10 tokens del vocabulario: {}".format(list(vocabulario.keys())[:10]))
print("Primeros 10 ids del vocabulario: {}".format(list(vocabulario.values())[:10]))
# Revisar si un token está presente en el vocabulario
token = "AAAAA"
print('¿Está {} en el vocabulario?: {}'.format(token, token in vocabulario.keys()))

# 3. Preparación de datos para el ajuste fino (_fine-tuning_)

## 3.1. Creación de clase PromotoresDataset
Esta clase se encarga de tokenizar las secuencias, obtener los token ids, attention_mask y tipos de promotor verdaderos.

In [ ]:
class PromotoresDataset(Dataset):
    def __init__(self, secuencias, tipos_promotor_verdaderos, tokenizador, num_token_maximo):
        #super(PromotoresDataset, self).__init__()
        self.secuencias = secuencias
        # tokeniza las secuencias
        salida_tokenizador = tokenizador(
            self.secuencias,
            # Agrega tokend de relleno hasta num_token_maximo
            padding="max_length",
            # Trunca tokens si pasan de num_token_maximo
            truncation=True,
            # Regresa token ids, attention_mask y tipo de promotor verdadero como tensores
            return_tensors="pt",
            # Define número de tokens máximo (num_token_maximo)
            max_length=num_token_maximo
        )
        # Obtiene token ids, attention_mask y tipo de promotor verdadero
        self.input_ids = salida_tokenizador["input_ids"]
        self.attention_mask = salida_tokenizador["attention_mask"]
        self.tipos_promotor_verdaderos = tipos_promotor_verdaderos

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        # Regresa token id, attention_mask y tipo de promotor verdadero de cada secuencia
        return dict(
            input_ids=self.input_ids[i],
            attention_mask=self.attention_mask[i],
            tipos_promotor_verdaderos=self.tipos_promotor_verdaderos[i]
        )


## 3.2. Creación de objetos Dataset
1.   dataset_train
2.   dataset_test



In [ ]:

dataset_train = PromotoresDataset(df_train['Sequence'].tolist(), df_train['Category'].tolist(), tokenizer, max_tokens)
dataset_test = PromotoresDataset(df_test['Sequence'].tolist(), df_test['Category'].tolist(), tokenizer, max_tokens)


------------------------------------------------------------------------


## 3.3. Creación de objetos DataLoader
1.   dataloader_train
2.   dataloader_test

Para definir los objetos DataLoader, es necesario establecer un tamaño de lote (`batch_size`). El _transfomer_ pre-entrenado DNABERT-2 mejorará su aprendizaje cada lote.



In [ ]:
batch_size = 4

dataloader_train = DataLoader(dataset_train, batch_size=batch_size)
dataloader_test = DataLoader(dataset_test, batch_size=batch_size, shuffle=False)


# 4. Definición del `device` (hardware) para ejecutar entrenamiento y prueba


In [ ]:
device = "cuda"

# 5. Ajuste fino del Modelo de Lenguaje Genómico (_fine-tuning_)
## 5.1. Descarga de DNABERT-2

Sólo es necesario indicar el nombre de DNABERT-2 y el número de tipos de promotor que vamos a clasificar. Automáticamente se descargará el modelo DNABERT-2 de la plataforma [Hugging Face](https://huggingface.co/).

In [ ]:
num_tipos_promotor = 2 # tipos de promotor = [1, 0]
red_transformer = BertForSequenceClassification.from_pretrained("zhihan1996/DNABERT-2-117M",
                                                                num_labels=num_tipos_promotor)



## 5.2. Envío de DNABERT-2 a device (cuda)
Permite que DNABERT-2 entrene en la GPU.

In [ ]:
red_transformer.to(device)

## 5.3. Definición de función de pérdida, optimizador y tasa de aprendizaje


* Función de pérdida (_loss function_): acumula el error en la predicción del modelo
  * Categoría verdadera del promotor - Categoría predicha por el modelo
  * Entropía cruzada (CrossEntropyLoss)
* Optimizador: ajusta los pesos (_w_) para disminuir el error
  * Optimizador adaptativo (Adam)
* Tasa de aprendizaje (learning rate o `lr`): controla el tamaño del ajuste de los pesos
  * 0.01 (`1e-2`).
  * 0.001 (`1e-3`)
  * 0.0001 (`1e-4`)
  * 0.00001 (`1e-5`)

In [ ]:
funcion_de_perdida = nn.CrossEntropyLoss()
optimizador_de_pesos = torch.optim.Adam(red_transformer.parameters(), lr=0.01)

## 5.4 Entrenamiento (_fine-tuning_)

### 5.4.1. Función de entrenamiento (`train`)

Esta función procesará las secuencias de cada lote y predicirá el tipo de promotor para cada una. Comparando el tipo predicho con el tipo verdadero calculará el valor de la función de pérdida. Luego ejecutará un algoritmo para analizar los pesos (retropropagación/_backpropagation_) y el optimizador para ajustarlos. A más lotes, DNABERT-2 deberá predecir mejor.



In [ ]:
def train(dataloader, red, funcion_perdida, optimizador):
    # Activamos nuestra red en modo de entrenamiento
    red.train()

    # Para cada lote de secuencias:
    for lote, X in enumerate(dataloader):
        # Enviar token ids, attention_mask y tipos de promotor verdaderos al device
        input_ids = X['input_ids'].to(device)
        input_mask = X['attention_mask'].to(device)
        tipos_promotor_verdaderos = X['tipos_promotor_verdaderos'].to(device)

        # Obtenemos predicciones de DNABERT-2 (logits) para las secuencias del lote
        logits = red(input_ids, input_mask).logits
        # Obtenemos valor de función de pérdida
        valor_de_perdida = funcion_perdida(logits, tipos_promotor_verdaderos)

        # Ejecución de backpropagation
        valor_de_perdida.backward()
        # Ajuste (modificación) de pesos
        optimizador.step()
        optimizador.zero_grad()

        if lote % 5 == 0:
            print("Lote:", lote, "- Valor de función de pérdida:", valor_de_perdida.item())

### 5.4.2. Ejecución de entrenamiento de DNABERT-2

Ejecutamos la función de entrenamiento (`train`) por un número de épocas (veces que la red neuronal procesa todos los datos de entrenamiento).

La función `train` imprimirá el valor de la función de pérdida cada 5 lotes

In [ ]:
epocas = 10
for epoca in range(epocas):
    print("\nFase de entrenamiento\n-------------------------------")
    print("Epoca", epoca+1)
    train(dataloader_train, red_transformer, funcion_de_perdida, optimizador_de_pesos)


# 6. Evaluación de la red y análisis de resultados
## 6.1 Función de prueba (`test`)

Esta función procesará las secuencias de cada lote y predicirá el tipo de promotor para cada una. Comparando el tipo predicho con el tipo verdadero calculará el valor de la función de pérdida. Además, juntará en una lista las tipos de promotor verdaderos y las tipos predichos por DNABERT-2. Estas listas serán usadas posteriormente para obtener métricas de evaluación y analizar resultados.

In [ ]:
def test(dataloader, red, funcion_perdida):
    # Activamos nuestra red en modo de prueba
    red.eval()

    # Creamos listas para juntar los tipos de promotor predichos y los tipos de promotor verdaderos
    lista_tipos_predichos = []
    lista_tipos_verdaderos = []

    # Desactivamos ajuste de pesos
    with torch.no_grad():
        # Para cada lote de secuencias:
        for lote, X in enumerate(dataloader):
            # Enviamos token ids, attention_mask y tipos de promotr verdaderos al device
            input_ids = X['input_ids'].to(device)
            input_mask = X['attention_mask'].to(device)
            tipos_promotor_verdaderos = X['tipos_promotor_verdaderos'].to(device)

            # Obtenemos predicciones de DNABERT-2 (logits) para las secuencias del lote
            logits = red(input_ids, input_mask).logits
            # Obtenemos el valor de la función de pérdida
            valor_de_perdida = funcion_de_perdida(logits, tipos_promotor_verdaderos)

            # Obtenemos tipos de promotor predichos
            # argmax() regresa el índice del valor máximo en el tensor.
            tipos_promotor_predichos = torch.argmax(logits, dim=1)

            # Juntamos en listas de forma acumulativa los tipos de promotor predichos y los tipos verdaderos
            lista_tipos_predichos.extend(tipos_promotor_predichos.tolist())
            lista_tipos_verdaderos.extend(tipos_promotor_verdaderos.tolist())

            if lote % 5 == 0:
              print("Lote:", lote, "- Valor de función de pérdida:", valor_de_perdida.item())

    return lista_tipos_predichos, lista_tipos_verdaderos


## 6.2 Ejecución de fase de prueba de DNABERT-2
Ejecutamos la función de prueba (`test`).

La función `test` imprimirá el valor de la función de pérdida cada 5 lotes

In [ ]:
print("\nFase de prueba\n-------------------------------")
tipos_promotor_predichos = []
tipos_promotor_verdaderos = []
tipos_promotor_predichos, tipos_promotor_verdaderos = test(dataloader_test, red_transformer, funcion_de_perdida)

### Obtención de métricas y reporte de clasificación

In [ ]:
# Obtenemos metricas de evaluación comparando las categorías verdaderas con las categorías predichas
print("F1-score: %0.2f" % f1_score(tipos_promotor_verdaderos, tipos_promotor_predichos, zero_division=0.0))
print("Precision: %0.2f" % precision_score(tipos_promotor_verdaderos, tipos_promotor_predichos, zero_division=0.0))
print("Recall: %0.2f" % recall_score(tipos_promotor_verdaderos, tipos_promotor_predichos, zero_division=0.0))
print("Área bajo la curva ROC: %0.2f" % roc_auc_score(tipos_promotor_verdaderos, tipos_promotor_predichos))
print("Reporte de clasificación")
print(classification_report(tipos_promotor_verdaderos, tipos_promotor_predichos, zero_division=0.0))


### Obtención de matriz de confusión normalizada

In [ ]:
matrix_confusion=confusion_matrix(tipos_promotor_verdaderos, tipos_promotor_predichos, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=matrix_confusion)
disp.plot(cmap=plt.cm.Blues)
plt.show()



---

# Si el tiempo lo permite...

Explorar un objeto DataLoader

In [ ]:
for X in dataloader_train:
    print("Dimensionalidad de datos del primer lote de entrenamiento:", X['input_ids'].shape)
    print("Ids de tokens del primer lote de entrenamiento:", X['input_ids'])
    print("Dimensionalidad de tipos de promotor del primer lote de entrenamiento:", X['tipos_promotor_verdaderos'].shape)
    print("Tipos de promotor del primer lote de entrenamiento:", X['tipos_promotor_verdaderos'])
    break